# 56. The Safety Stock & Reorder Point (Q,r) Policy Problem

## Tier 1: Mathematical Formulation

### Key Assumptions
- Demand follows a normal distribution with known mean and standard deviation
- Lead time is stochastic with known mean and standard deviation
- Stockout costs are linear in the number of units short
- Holding costs are linear in the average inventory level
- Ordering costs are fixed per order regardless of order size

### Approach (Step-by-Step)
1. **Model the inventory system as a network flow problem** where inventory flows through time and states
2. **Formulate the objective function** to minimize expected total costs (holding + ordering + stockout)
3. **Define constraints** for flow conservation, reorder point logic, and non-negativity
4. **Calculate safety stock** using the combined uncertainty of demand and lead time
5. **Determine reorder point** as expected lead time demand plus safety stock
6. **Compute economic order quantity** using the classic EOQ formula

### What to Look For in the Results
- Optimal safety stock level that provides the required service level
- Reorder point that triggers orders when inventory reaches the threshold
- Economic order quantity that balances ordering and holding costs
- Total cost breakdown showing the trade-offs between different cost components

### Concrete Example: CardioStabil Medication
- Annual demand: 12,000 units (CV = 0.3)
- Daily demand: μ = 33 units, σ = 10 units
- Lead time: L = 14 days, σ = 3 days
- Unit cost: $450 per vial
- Holding cost rate: 25% per year
- Ordering cost: $200 per order
- Stockout cost: $2,000 per unit
- Required service level: 99.5% (z = 2.576)

In [1]:
# Import required libraries for mathematical formulation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import math

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for mathematical formulation")

Libraries imported successfully for mathematical formulation


Libraries imported successfully for mathematical formulation


Libraries imported successfully for mathematical formulation


Libraries imported successfully for mathematical formulation


Libraries imported successfully for mathematical formulation


In [ ]:
# Define the CardioStabil problem parameters
class InventoryParameters:
    """Data class for inventory problem parameters"""
    def __init__(self):
        # Demand parameters
        self.annual_demand = 12000  # units per year
        self.daily_demand_mean = self.annual_demand / 365  # 33 units per day
        self.demand_cv = 0.3  # coefficient of variation
        self.daily_demand_std = self.daily_demand_mean * self.demand_cv  # 10 units
        
        # Lead time parameters
        self.lead_time_mean = 14  # days
        self.lead_time_std = 3  # days
        
        # Cost parameters
        self.unit_cost = 450  # $ per vial
        self.holding_cost_rate = 0.25  # 25% per year
        self.holding_cost_per_unit = self.unit_cost * self.holding_cost_rate  # $112.5 per year
        self.ordering_cost = 200  # $ per order
        self.stockout_cost = 2000  # $ per unit short
        
        # Service level
        self.service_level = 0.995  # 99.5% service level
        self.z_score = stats.norm.ppf(self.service_level)  # 2.576

# Initialize parameters
params = InventoryParameters()

# Display key parameters
print("=== CardioStabil Inventory Parameters ===")
print(f"Annual Demand: {params.annual_demand:,} units")
print(f"Daily Demand: μ = {params.daily_demand_mean:.1f}, σ = {params.daily_demand_std:.1f} units")
print(f"Lead Time: μ = {params.lead_time_mean} days, σ = {params.lead_time_std} days")
print(f"Unit Cost: ${params.unit_cost:,}")
print(f"Holding Cost: ${params.holding_cost_per_unit:.2f} per unit per year")
print(f"Ordering Cost: ${params.ordering_cost:,} per order")
print(f"Stockout Cost: ${params.stockout_cost:,} per unit")
print(f"Service Level: {params.service_level*100:.1f}% (z = {params.z_score:.3f})")

In [ ]:
# Mathematical formulation: Calculate safety stock using combined uncertainty
def calculate_safety_stock(params):
    """
    Calculate safety stock considering both demand and lead time uncertainty.
    
    Formula: SS = z * sqrt(L * σ_D^2 + μ^2 * σ_L^2)
    where:
    - z: z-score for service level
    - L: mean lead time
    - σ_D: demand standard deviation
    - μ: mean demand
    - σ_L: lead time standard deviation
    """
    # Calculate variance components
    demand_variance = params.lead_time_mean * (params.daily_demand_std ** 2)
    lead_time_variance = (params.daily_demand_mean ** 2) * (params.lead_time_std ** 2)
    
    # Combined standard deviation
    combined_std = math.sqrt(demand_variance + lead_time_variance)
    
    # Safety stock calculation
    safety_stock = params.z_score * combined_std
    
    return safety_stock, combined_std, demand_variance, lead_time_variance

# Calculate safety stock for CardioStabil
safety_stock, combined_std, demand_var, lead_time_var = calculate_safety_stock(params)

print("=== Safety Stock Calculation ===")
print(f"Demand Variance Component: {demand_var:.2f}")
print(f"Lead Time Variance Component: {lead_time_var:.2f}")
print(f"Combined Standard Deviation: {combined_std:.2f} units")
print(f"Z-Score for {params.service_level*100:.1f}% Service Level: {params.z_score:.3f}")
print(f"Safety Stock: {safety_stock:.1f} units")
print(f"Rounded Safety Stock: {round(safety_stock)} units")

In [ ]:
# Calculate reorder point and economic order quantity
def calculate_reorder_point(params, safety_stock):
    """
    Calculate reorder point: r = μ * L + SS
    """
    expected_lead_time_demand = params.daily_demand_mean * params.lead_time_mean
    reorder_point = expected_lead_time_demand + safety_stock
    return reorder_point, expected_lead_time_demand

def calculate_economic_order_quantity(params):
    """
    Calculate Economic Order Quantity (EOQ):
    Q* = sqrt(2 * D * K / h)
    where:
    - D: annual demand
    - K: ordering cost per order
    - h: holding cost per unit per year
    """
    eoq = math.sqrt((2 * params.annual_demand * params.ordering_cost) / params.holding_cost_per_unit)
    return eoq

# Calculate reorder point and EOQ
reorder_point, expected_lt_demand = calculate_reorder_point(params, safety_stock)
eoq = calculate_economic_order_quantity(params)

print("=== Reorder Point Calculation ===")
print(f"Expected Lead Time Demand: {expected_lt_demand:.1f} units")
print(f"Safety Stock: {safety_stock:.1f} units")
print(f"Reorder Point: {reorder_point:.1f} units")
print(f"Rounded Reorder Point: {round(reorder_point)} units")

print("\n=== Economic Order Quantity Calculation ===")
print(f"Annual Demand: {params.annual_demand:,} units")
print(f"Ordering Cost: ${params.ordering_cost:,} per order")
print(f"Holding Cost per Unit: ${params.holding_cost_per_unit:.2f} per year")
print(f"EOQ: {eoq:.1f} units")
print(f"Rounded EOQ: {round(eoq)} units")

In [ ]:
# Calculate total cost components for the optimal policy
def calculate_total_costs(params, order_quantity, reorder_point, safety_stock):
    """
    Calculate total annual cost components:
    - Holding cost: (cycle stock + safety stock) * holding cost per unit
    - Ordering cost: (annual demand / order quantity) * ordering cost
    - Stockout cost: expected stockouts * stockout cost per unit
    """
    # Holding cost calculation
    cycle_stock = order_quantity / 2
    average_inventory = cycle_stock + safety_stock
    holding_cost = average_inventory * params.holding_cost_per_unit
    
    # Ordering cost calculation
    orders_per_year = params.annual_demand / order_quantity
    ordering_cost = orders_per_year * params.ordering_cost
    
    # Stockout cost calculation (simplified approximation)
    stockout_probability = 1 - params.service_level
    expected_stockouts_per_cycle = stockout_probability * params.annual_demand / (365 / params.lead_time_mean)
    stockout_cost = expected_stockouts_per_cycle * params.stockout_cost
    
    # Total cost
    total_cost = holding_cost + ordering_cost + stockout_cost
    
    return {
        'holding_cost': holding_cost,
        'ordering_cost': ordering_cost,
        'stockout_cost': stockout_cost,
        'total_cost': total_cost,
        'cycle_stock': cycle_stock,
        'average_inventory': average_inventory,
        'orders_per_year': orders_per_year
    }

# Calculate costs for the optimal policy
optimal_q = round(eoq)
optimal_r = round(reorder_point)
optimal_ss = round(safety_stock)

costs = calculate_total_costs(params, optimal_q, optimal_r, optimal_ss)

print("=== Total Cost Analysis ===")
print(f"Optimal Policy: (Q = {optimal_q}, r = {optimal_r}, SS = {optimal_ss})")
print(f"Cycle Stock: {costs['cycle_stock']:.1f} units")
print(f"Average Inventory: {costs['average_inventory']:.1f} units")
print(f"Orders per Year: {costs['orders_per_year']:.1f}")
print(f"Holding Cost: ${costs['holding_cost']:,.2f}")
print(f"Ordering Cost: ${costs['ordering_cost']:,.2f}")
print(f"Stockout Cost: ${costs['stockout_cost']:,.2f}")
print(f"Total Annual Cost: ${costs['total_cost']:,.2f}")

In [ ]:
# Create comprehensive visualization of the (Q,r) policy
def create_policy_visualization(params, optimal_q, optimal_r, optimal_ss, costs):
    """
    Create a 4-panel visualization showing:
    1. Inventory level over time with reorder point
    2. Cost breakdown pie chart
    3. Safety stock components
    4. Sensitivity analysis
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Safety Stock & Reorder Point (Q,r) Policy Analysis - CardioStabil', fontsize=16, fontweight='bold')
    
    # Panel 1: Inventory Level Over Time
    time_periods = 60  # Show 60 days of inventory dynamics
    time = np.arange(0, time_periods)
    inventory_level = []
    current_inventory = optimal_r + optimal_q  # Start with full inventory
    
    for t in time:
        if current_inventory <= optimal_r and t > 0:
            current_inventory += optimal_q  # Order arrives
        demand = np.random.normal(params.daily_demand_mean, params.daily_demand_std)
        current_inventory = max(0, current_inventory - demand)
        inventory_level.append(current_inventory)
    
    ax1.plot(time, inventory_level, 'b-', linewidth=2, label='Inventory Level')
    ax1.axhline(y=optimal_r, color='r', linestyle='--', linewidth=2, label=f'Reorder Point ({optimal_r})')
    ax1.axhline(y=optimal_ss, color='g', linestyle=':', linewidth=2, label=f'Safety Stock ({optimal_ss})')
    ax1.fill_between(time, 0, optimal_ss, alpha=0.3, color='green', label='Safety Stock Buffer')
    ax1.set_xlabel('Time (days)')
    ax1.set_ylabel('Inventory Level (units)')
    ax1.set_title('Inventory Level Dynamics Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Cost Breakdown
    cost_components = ['Holding Cost', 'Ordering Cost', 'Stockout Cost']
    cost_values = [costs['holding_cost'], costs['ordering_cost'], costs['stockout_cost']]
    colors = ['#ff9999', '#66b3ff', '#99ff99']
    
    wedges, texts, autotexts = ax2.pie(cost_values, labels=cost_components, colors=colors, autopct='%1.1f%%', 
                                      startangle=90, textprops={'fontsize': 10})
    ax2.set_title(f'Annual Cost Breakdown\nTotal: ${costs["total_cost"]:,.0f}')
    
    # Panel 3: Safety Stock Components
    ss_components = ['Demand Uncertainty', 'Lead Time Uncertainty']
    ss_values = [demand_var, lead_time_var]
    
    bars = ax3.bar(ss_components, ss_values, color=['#ff6b6b', '#4ecdc4'])
    ax3.set_ylabel('Variance')
    ax3.set_title('Safety Stock Variance Components')
    
    # Add value labels on bars
    for bar, value in zip(bars, ss_values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{value:.0f}', ha='center', va='bottom', fontweight='bold')
    
    # Panel 4: Service Level Sensitivity
    service_levels = np.linspace(0.90, 0.999, 20)
    safety_stocks = []
    
    for sl in service_levels:
        z = stats.norm.ppf(sl)
        ss = z * combined_std
        safety_stocks.append(ss)
    
    ax4.plot(service_levels * 100, safety_stocks, 'purple', linewidth=2)
    ax4.scatter([params.service_level * 100], [optimal_ss], color='red', s=100, zorder=5, 
               label=f'Current: {params.service_level*100:.1f}%')
    ax4.set_xlabel('Service Level (%)')
    ax4.set_ylabel('Safety Stock (units)')
    ax4.set_title('Safety Stock vs Service Level')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create visualization
create_policy_visualization(params, optimal_q, optimal_r, optimal_ss, costs)

In [ ]:
# Sensitivity analysis: How optimal parameters change with key inputs
def sensitivity_analysis(params):
    """
    Perform sensitivity analysis on key parameters:
    - Demand variability
    - Lead time variability
    - Service level
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Sensitivity Analysis of (Q,r) Policy Parameters', fontsize=14, fontweight='bold')
    
    # Sensitivity to demand variability
    demand_cvs = np.linspace(0.1, 0.5, 10)
    ss_demand = []
    rp_demand = []
    
    for cv in demand_cvs:
        temp_params = InventoryParameters()
        temp_params.demand_cv = cv
        temp_params.daily_demand_std = temp_params.daily_demand_mean * cv
        
        ss, _, _, _ = calculate_safety_stock(temp_params)
        rp, _ = calculate_reorder_point(temp_params, ss)
        
        ss_demand.append(ss)
        rp_demand.append(rp)
    
    ax1.plot(demand_cvs * 100, ss_demand, 'b-', linewidth=2, marker='o')
    ax1.set_xlabel('Demand Coefficient of Variation (%)')
    ax1.set_ylabel('Safety Stock (units)')
    ax1.set_title('Safety Stock vs Demand Variability')
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(demand_cvs * 100, rp_demand, 'r-', linewidth=2, marker='s')
    ax2.set_xlabel('Demand Coefficient of Variation (%)')
    ax2.set_ylabel('Reorder Point (units)')
    ax2.set_title('Reorder Point vs Demand Variability')
    ax2.grid(True, alpha=0.3)
    
    # Sensitivity to lead time variability
    lead_time_stds = np.linspace(1, 7, 10)
    ss_lt = []
    rp_lt = []
    
    for lt_std in lead_time_stds:
        temp_params = InventoryParameters()
        temp_params.lead_time_std = lt_std
        
        ss, _, _, _ = calculate_safety_stock(temp_params)
        rp, _ = calculate_reorder_point(temp_params, ss)
        
        ss_lt.append(ss)
        rp_lt.append(rp)
    
    ax3.plot(lead_time_stds, ss_lt, 'g-', linewidth=2, marker='^')
    ax3.set_xlabel('Lead Time Standard Deviation (days)')
    ax3.set_ylabel('Safety Stock (units)')
    ax3.set_title('Safety Stock vs Lead Time Variability')
    ax3.grid(True, alpha=0.3)
    
    ax4.plot(lead_time_stds, rp_lt, 'm-', linewidth=2, marker='d')
    ax4.set_xlabel('Lead Time Standard Deviation (days)')
    ax4.set_ylabel('Reorder Point (units)')
    ax4.set_title('Reorder Point vs Lead Time Variability')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Run sensitivity analysis
sensitivity_analysis(params)

In [ ]:
# Summary and key insights
print("=== MATHEMATICAL FORMULATION SUMMARY ===")
print("\n📊 OPTIMAL POLICY PARAMETERS:")
print(f"   • Order Quantity (Q): {optimal_q:,} units")
print(f"   • Reorder Point (r): {optimal_r:,} units")
print(f"   • Safety Stock: {optimal_ss:,} units")
print(f"   • Service Level: {params.service_level*100:.1f}%")

print("\n💰 COST BREAKDOWN:")
print(f"   • Holding Cost: ${costs['holding_cost']:,.2f} ({costs['holding_cost']/costs['total_cost']*100:.1f}%)")
print(f"   • Ordering Cost: ${costs['ordering_cost']:,.2f} ({costs['ordering_cost']/costs['total_cost']*100:.1f}%)")
print(f"   • Stockout Cost: ${costs['stockout_cost']:,.2f} ({costs['stockout_cost']/costs['total_cost']*100:.1f}%)")
print(f"   • Total Annual Cost: ${costs['total_cost']:,.2f}")

print("\n🔢 KEY MATHEMATICAL FORMULAS:")
print("   • Safety Stock: SS = z × √(L × σ²_D + μ² × σ²_L)")
print("   • Reorder Point: r = μ × L + SS")
print("   • Economic Order Quantity: Q* = √(2 × D × K / h)")

print("\n📈 PERFORMANCE METRICS:")
print(f"   • Orders per Year: {costs['orders_per_year']:.1f}")
print(f"   • Average Inventory: {costs['average_inventory']:.1f} units")
print(f"   • Cycle Stock: {costs['cycle_stock']:.1f} units")
print(f"   • Inventory Investment: ${costs['average_inventory'] * params.unit_cost:,.2f}")

print("\n✅ MATHEMATICAL FORMULATION COMPLETE")
print("The mathematical formulation provides the exact optimal solution")
print("for the (Q,r) inventory policy under uncertainty assumptions.")